# 第15章 データクレンジング (実務編)

実務では、初めから「きれいなデータ」が提供されることはほぼありません。
このノートブックでは、現場でよく出会う **データの汚れ** と **その整え方** を学びます。

## このノートブックで扱う典型的な「汚れ」

| # | 種類 | 例 |
|-|-|-|
| ① | **表記ゆれ (大文字小文字)** | `Apple`, `apple`, `APPLE` |
| ② | **余計な空白** | `' 商品A '`, `'  田中  '` |
| ③ | **全角と半角の混在** | `'１２３'` と `'123'`, `'Ａ'` と `'A'` |
| ④ | **日付フォーマットの揺れ** | `'2024/01/01'`, `'2024-01-01'`, `'1月1日'` |
| ⑤ | **Excel のシリアル値** (数値で日付を表現) | `45292` ← 実は 2024-01-01 |
| ⑥ | **数値が文字列で混じる** | `'1500'`, `'1,500'`, `'¥1500'`, `'１５００'` |
| ⑦ | **欠損値** (NaN / None / 空文字) | 章 10 を参照 |

## 学習目標

- pandas の `.str` アクセサで文字列を整形できる
- `astype` や `to_datetime` で型変換できる
- 全角半角を変換できる
- 集計結果が変わることを **クレンジング前後で比較** できる


## 15.1 import

In [ ]:
import numpy as np
import pandas as pd

# 全角⇔半角の変換に使う (要 pip install jaconv)
try:
    import jaconv
    HAS_JACONV = True
except ImportError:
    HAS_JACONV = False
    print('jaconv が未インストール → `pip install jaconv` で入れると全角変換セクションが動きます')


## 15.2 サンプルデータの準備

実務でありがちな「汚れ」を含むデータを意図的に作ります。


In [ ]:
raw = pd.DataFrame({
    'product':        ['商品A', '商品 A', 'apple', 'Apple', 'APPLE ', '商品B', '商品 b', '商品B'],
    'customer':       ['田中', ' 田中', '佐藤', '鈴木', '田中', '佐藤', '鈴木 ', '田中'],
    'purchase_date':  ['2024/01/05', '2024-01-05', '2024年1月10日', '45298', '2024/02/01', '20240202', '2024-02-15', '2024-02-15'],
    'price':          ['1500', '1,500', '￥2000', '２０００', '3000', '3,000円', '1500', np.nan],
})
raw


このまま集計するとどうなるかを見てみます。


In [ ]:
# 表記ゆれがあるため、同じ商品なのに別物として集計されてしまう
print('=== クレンジング前: 商品ごとの行数 ===')
print(raw['product'].value_counts())
print()
print('=== クレンジング前: 顧客ごとの行数 ===')
print(raw['customer'].value_counts())


## 15.3 ① 表記ゆれの統一 (`.str` アクセサ)

pandas の **`.str` アクセサ** を使うと、Series 全体に対して文字列操作ができます。

| メソッド | 用途 |
|-|-|
| `.str.lower()` / `.str.upper()` | 大文字小文字を統一 |
| `.str.strip()` | 前後の空白を除去 |
| `.str.replace(古, 新)` | 部分置換 |
| `.str.split(sep)` | 分割 |
| `.str.contains(pat)` | 含むかどうかの判定 |


In [ ]:
df = raw.copy()

# product 列をクレンジング
df['product'] = (
    df['product']
    .str.strip()                       # 前後の空白を取る
    .str.replace(' ', '', regex=False) # 半角スペースを削除
    .str.replace('　', '', regex=False)# 全角スペースを削除
    .str.upper()                       # すべて大文字に統一
)
print(df['product'].value_counts())


**観察**: 8 行を 4 種類 (`商品A`, `商品B`, `APPLE`) にまとめられました。


## 15.4 ② 顧客名の表記ゆれ

同じ要領で `customer` 列もクレンジングします。


In [ ]:
df['customer'] = (
    df['customer']
    .str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('　', '', regex=False)
)
print(df['customer'].value_counts())


## 15.5 ③ 全角⇔半角の変換

`jaconv` ライブラリを使うと、日本語の全角と半角を相互変換できます。

```bash
pip install jaconv
```

| 関数 | 用途 |
|-|-|
| `jaconv.z2h(s)` | 全角 → 半角 |
| `jaconv.h2z(s)` | 半角 → 全角 |
| `jaconv.z2h(s, kana=False, digit=True, ascii=True)` | 引数で対象 (かな・数字・英字) を絞れる |


In [ ]:
if HAS_JACONV:
    print('全角数字 → 半角:', jaconv.z2h('１２３４５', kana=False, digit=True, ascii=False))
    print('全角英字 → 半角:', jaconv.z2h('ＡＢＣ', kana=False, digit=False, ascii=True))
    print('半角カナ → 全角:', jaconv.h2z('ｱｲｳｴｵ'))
else:
    print('jaconv 未インストールのためスキップ')


## 15.6 ④ 価格列のクレンジング

`price` には `'¥1500'`, `'1,500'`, `'２０００'`, `'3000円'` などが混在しています。
これを統一して **数値列** にします。


In [ ]:
def clean_price(value):
    """価格文字列を整数に変換する。変換できなければ NaN を返す。"""
    if pd.isna(value):
        return np.nan
    s = str(value)
    # 全角 → 半角 (英数字)
    if HAS_JACONV:
        s = jaconv.z2h(s, kana=False, digit=True, ascii=True)
    # 余計な文字を除去
    for ch in ['¥', '￥', '円', ',', ' ', '　']:
        s = s.replace(ch, '')
    try:
        return int(s)
    except ValueError:
        return np.nan

df['price_clean'] = df['price'].apply(clean_price)
df[['price', 'price_clean']]


## 15.7 ⑤ 日付の統一 (to_datetime とシリアル値)

`pd.to_datetime` は **柔軟なパーサ** で、多くの形式を自動認識してくれます。
ただし、Excel のシリアル値 (`45298` のような数値) は別途処理が必要です。


In [ ]:
def parse_date(value):
    """いろいろな形式の日付文字列を date 型に変換する。"""
    if pd.isna(value):
        return pd.NaT
    s = str(value).strip()

    # Excel シリアル値 (数値だけ) → 1900-01-01 起算
    if s.isdigit() and len(s) <= 6:
        # Excel は 1900-01-01 を 1 として、さらに歴史的バグで +2 する必要がある
        return pd.to_datetime('1899-12-30') + pd.to_timedelta(int(s), unit='D')

    # 「2024年1月10日」のような和暦表記 → 区切り文字を共通化
    s = (
        s.replace('年', '-')
         .replace('月', '-')
         .replace('日', '')
         .replace('/', '-')
    )

    # 残りは pandas のパーサに任せる
    return pd.to_datetime(s, errors='coerce')

df['purchase_date_clean'] = df['purchase_date'].apply(parse_date)
df[['purchase_date', 'purchase_date_clean']]


**観察**: 各種フォーマットがすべて `Timestamp` 型に揃いました。
シリアル値 `45298` も正しく日付として解釈されています。


## 15.8 クレンジング後に集計しなおす

クレンジング前後で集計結果が変わることを確認します。


In [ ]:
# 商品ごとの売上 (= 件数 × 価格平均、簡易デモ用)
print('=== クレンジング後: 商品ごとの平均価格 ===')
print(df.groupby('product')['price_clean'].mean())
print()
print('=== クレンジング後: 顧客ごとの購入回数 ===')
print(df['customer'].value_counts())


## 15.9 重要: 「なぜ汚れているか」 を考える

機械的にクレンジングする前に、**なぜそのような汚れが入ったか** を確認することが重要です。

| ケース | 対応 |
|-|-|
| 単純な入力ミス | 自動修正 OK |
| ユーザーごとに入力ルールが違う | クレンジングルールを文書化, レビュー必須 |
| 別システムからの取り込み | 元システムでの修正を依頼 |
| データ自体に意味がある | 機械的に変えない (例: 「○○ (旧)」と「○○ (新)」を別物として扱いたいケース) |

> クレンジングは「データを安易に書き換える」と元データの情報を失います。
> **元データは必ず残し、加工後のものを別カラム/別ファイルとして保存** するのが基本です。


## 15.10 クレンジング全体のチェックリスト

新しいデータを受け取ったら、まずこのチェックリストを当てて見るのがおすすめ:

- [ ] 行数・列数は想定どおり?
- [ ] 各列のデータ型は妥当?
- [ ] 欠損値はどこに、どれくらいある?
- [ ] 文字列の **大文字小文字, 全角半角, 前後空白** はそろってる?
- [ ] 数値列に文字が混じってない?
- [ ] 日付列はパースできる?
- [ ] **重複行** はある? (`drop_duplicates`)
- [ ] 値の範囲は妥当? (年齢が負の値, 価格 0 円など)
- [ ] カテゴリ列のユニーク値は想定どおり? (`value_counts`)


## 15.11 練習問題


In [ ]:
# Q1. このデータを集計可能な形にクレンジングしよう
dirty = pd.DataFrame({
    'name':  ['ALICE', ' alice', 'Bob', 'bob ', 'CAROL'],
    'price': ['1000', '1,000円', '２０００', '3000', '￥4000'],
})
print(dirty)
print('\nクレンジング前の name value_counts:')
print(dirty['name'].value_counts())


In [ ]:
# 解答例: name と price を綺麗に整える
clean = dirty.copy()

clean['name'] = (
    clean['name']
    .str.strip()
    .str.replace(' ', '', regex=False)
    .str.lower()
)

def to_int_price(s):
    s = str(s)
    if HAS_JACONV:
        s = jaconv.z2h(s, kana=False, digit=True, ascii=True)
    for ch in ['¥', '￥', '円', ',', ' ']:
        s = s.replace(ch, '')
    try:
        return int(s)
    except ValueError:
        return np.nan

clean['price'] = clean['price'].apply(to_int_price)

print(clean)
print('\nクレンジング後の name value_counts:')
print(clean['name'].value_counts())
print('\n合計売上:', clean['price'].sum())


## まとめ

- 文字列の整形は **`.str` アクセサ** を活用 (`strip`, `replace`, `lower`, `upper`)
- 全角/半角は **`jaconv`** を使うと簡単
- 日付は **`pd.to_datetime`** で柔軟にパース、Excel シリアル値は別途計算
- 機械的にクレンジングする前に **「なぜ汚れているか」** を確認
- 元データは保存して、加工結果は別カラム/別ファイルへ
- 新データを触る前に **チェックリスト** を当てる習慣を
